In [1]:
# Get working dir right

import os 

while 'TSDP' != os.getcwd().split('\\')[-1]:
    os.chdir("..")
    
print(os.getcwd())

c:\Users\Adam\..Data\TSDP


In [2]:
# Define save function

import os
import pandas as pd
from typing import Dict, Union, List

def save_xlsx(df: Union[pd.DataFrame, Dict[str, pd.DataFrame]], 
              folder: str, 
              filename: str, 
              ask_before_overwrite: bool = True,
              sheet_name: str = None) -> str:
    """
    DataFrame vagy DataFrame-ek szótárának elmentése Excelbe egy megadott mappába.
    
    Args:
        df: pandas DataFrame vagy munkalapnév-DataFrame párok szótára
        folder: mappa elérési út (str)
        filename: fájlnév .xlsx kiterjesztéssel (str)
        ask_before_overwrite: ha True, felülírás előtt megkérdezi a usert
        sheet_name: munkalap neve (ha egy DataFrame-t mentünk)
    
    Returns:
        saved_path (str) – a létrehozott fájl teljes elérési útja
    """
    os.makedirs(folder, exist_ok=True)  # ha nem létezik, létrehozza
    save_path = os.path.join(folder, filename)
    
    if os.path.exists(save_path) and ask_before_overwrite:
        resp = input(f"A fájl már létezik: {save_path}. Felülírjam? (y/n): ").strip().lower()
        if resp != "y":
            print("Mentés megszakítva.")
            return None
    
    # ExcelWriter létrehozása
    with pd.ExcelWriter(save_path, engine='openpyxl') as writer:
        if isinstance(df, dict):
            # Több munkalap mentése
            for sheet_name, sheet_df in df.items():
                sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
                print(f"Munkalap mentve: '{sheet_name}'")
        else:
            # Egy munkalap mentése
            actual_sheet_name = sheet_name if sheet_name else 'Sheet1'
            df.to_excel(writer, sheet_name=actual_sheet_name, index=False)
            print(f"Munkalap mentve: '{actual_sheet_name}'")
    
    print(f"Mentve ide: {save_path}")
    return save_path


def load_xlsx(filepath: str, sheet_name: str = None) -> Union[pd.DataFrame, Dict[str, pd.DataFrame]]:
    """
    Excel fájl betöltése DataFrame-be vagy DataFrame-ek szótárába.
    
    Args:
        filepath: teljes elérési út a fájlhoz (.xlsx)
        sheet_name: konkrét munkalap neve (ha None, akkor mindet betölti)
    
    Returns:
        DataFrame vagy munkalapnév-DataFrame párok szótára
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Nem található: {filepath}")
    
    if sheet_name:
        # Csak egy specifikus munkalap betöltése
        df = pd.read_excel(filepath, sheet_name=sheet_name)
        print(f"Betöltve: {filepath} (munkalap: '{sheet_name}')")
        return df
    else:
        # Összes munkalap betöltése
        all_sheets = pd.read_excel(filepath, sheet_name=None)
        print(f"Betöltve: {filepath} ({len(all_sheets)} munkalap)")
        return all_sheets

# FETCH DATA

In [17]:
# FIFA rankings

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import pandas as pd
import time
from datetime import datetime, timedelta

def setup_driver():
    """Set up Chrome driver with options"""
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36')
    
    driver = webdriver.Chrome(options=chrome_options)
    return driver

def scrape_football_ranking_period(period=None):
    """Scrape FIFA rankings from football-ranking.com for a specific period"""
    base_url = "https://football-ranking.com/fifa-rankings"
    driver = setup_driver()
    all_rankings_data = []
    
    try:
        # Construct URL with period parameter if provided
        if period:
            url = f"{base_url}?period={period.replace(' ', '+')}"
        else:
            url = base_url
        
        print(f"Accessing: {url}")
        driver.get(url)
        
        # Wait for table to load
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "table"))
            )
        except TimeoutException:
            print("Timeout waiting for table to load")
            return pd.DataFrame()
        
        # Get total number of pages (first 4 pages)
        total_pages = 4
        
        # Scrape each page
        for page in range(1, total_pages + 1):
            if page > 1:
                page_url = f"{url}&page={page}" if "?" in url else f"{url}?page={page}"
                driver.get(page_url)
                time.sleep(2)
            
            print(f"Scraping page {page}/{total_pages}...")
            
            # Wait for table to load
            try:
                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.TAG_NAME, "table"))
                )
            except TimeoutException:
                print(f"Timeout waiting for table on page {page}")
                continue
            
            # Scrape current page
            page_data = scrape_ranking_page(driver)
            all_rankings_data.extend(page_data)
            print(f"Found {len(page_data)} countries on page {page}")
        
        if not all_rankings_data:
            print("No ranking data found")
            return pd.DataFrame()
            
        df = pd.DataFrame(all_rankings_data)
        period_label = period if period else "current"
        print(f"Total rankings scraped for {period_label}: {len(df)} countries")
        return df
        
    except Exception as e:
        print(f"Error scraping football-ranking.com: {e}")
        return pd.DataFrame()
    finally:
        driver.quit()

def scrape_ranking_page(driver):
    """Scrape ranking data from current page"""
    rankings_data = []
    
    try:
        table = driver.find_element(By.TAG_NAME, "table")
        rows = table.find_elements(By.TAG_NAME, "tr")[1:]  # Skip header row
        
        for row in rows:
            try:
                # Skip ad rows
                if "script" in row.get_attribute("innerHTML"):
                    continue
                
                # Get all cells in the row
                cells = row.find_elements(By.TAG_NAME, "td")
                if len(cells) < 5:  # Should have at least 5 columns
                    continue
                
                # Rank (first column)
                rank_text = cells[0].text.strip()
                # Extract only the number from rank (handle cases like "51 (↓3)")
                rank = ''.join(filter(str.isdigit, rank_text.split()[0]))
                if not rank:
                    continue
                
                # Country (second column)
                country_elem = cells[1]
                country_text = country_elem.text.strip()
                # Extract country name (before any parentheses)
                country = country_text.split('(')[0].strip()
                
                # Points (third column)
                points_text = cells[2].text.strip()
                # Extract numeric points (remove commas and any extra text)
                points = ''.join(filter(lambda x: x.isdigit() or x == '.', points_text.split()[0]))
                
                # Previous points (fourth column, hidden on mobile)
                prev_points = cells[3].text.strip() if len(cells) > 3 else "N/A"
                
                # Previous rank (fifth column, hidden on mobile)
                prev_rank = cells[4].text.strip() if len(cells) > 4 else "N/A"
                
                rankings_data.append({
                    'Rank': int(rank),
                    'Country': country,
                    'Points': float(points) if points else 0,
                    'Previous_Points': prev_points,
                    'Previous_Rank': prev_rank
                })
                
            except Exception as e:
                print(f"Error processing row: {e}")
                continue
                
    except Exception as e:
        print(f"Error scraping page: {e}")
    
    return rankings_data

def save_xlsx(df, folder, filename):
    """Save DataFrame to Excel file"""
    import os
    os.makedirs(folder, exist_ok=True)
    filepath = os.path.join(folder, filename)
    df.to_excel(filepath, index=False)
    print(f"Data saved to {filepath}")

def main():
    """Main function to scrape both current and historical rankings"""
    # Scrape current rankings
    print("=== SCRAPING CURRENT FIFA RANKINGS ===")
    current_df = scrape_football_ranking_period()
    
    if not current_df.empty:
        save_xlsx(current_df, folder="HUN-ARM/data", filename="fifa_rankings_current.xlsx")
        
        # Find Hungary and Armenia in current rankings
        hungary_current = current_df[current_df['Country'].str.contains('Hungary|Magyar', case=False, na=False)]
        armenia_current = current_df[current_df['Country'].str.contains('Armenia|Örmény', case=False, na=False)]
        
        print("\n=== CURRENT RANKINGS ===")
        print("Hungary:", hungary_current['Rank'].values[0] if not hungary_current.empty else "Not found")
        print("Armenia:", armenia_current['Rank'].values[0] if not armenia_current.empty else "Not found")
        
        if not hungary_current.empty:
            print("Hungary details:", hungary_current.to_dict('records')[0])
        if not armenia_current.empty:
            print("Armenia details:", armenia_current.to_dict('records')[0])
    
    # Scrape historical rankings (August 2021)
    print("\n=== SCRAPING HISTORICAL FIFA RANKINGS (AUGUST 2021) ===")
    historical_period = "12 August 2021"
    historical_df = scrape_football_ranking_period(period=historical_period)
    
    if not historical_df.empty:
        save_xlsx(historical_df, folder="HUN-ARM/data", filename="fifa_rankings_historical.xlsx")
        
        # Find Hungary and Armenia in historical rankings
        hungary_historical = historical_df[historical_df['Country'].str.contains('Hungary|Magyar', case=False, na=False)]
        armenia_historical = historical_df[historical_df['Country'].str.contains('Armenia|Örmény', case=False, na=False)]
        
        print("\n=== HISTORICAL RANKINGS (AUGUST 2021) ===")
        print("Hungary:", hungary_historical['Rank'].values[0] if not hungary_historical.empty else "Not found")
        print("Armenia:", armenia_historical['Rank'].values[0] if not armenia_historical.empty else "Not found")
        
        if not hungary_historical.empty:
            print("Hungary details:", hungary_historical.to_dict('records')[0])
        if not armenia_historical.empty:
            print("Armenia details:", armenia_historical.to_dict('records')[0])
    
    # Compare rankings if both datasets are available
    if not current_df.empty and not historical_df.empty:
        print("\n=== RANKING COMPARISON (CURRENT vs AUGUST 2021) ===")
        
        # Hungary comparison
        if not hungary_current.empty and not hungary_historical.empty:
            hungary_change = hungary_historical['Rank'].values[0] - hungary_current['Rank'].values[0]
            direction = "improved" if hungary_change > 0 else "worsened" if hungary_change < 0 else "unchanged"
            print(f"Hungary: {hungary_current['Rank'].values[0]} (current) vs {hungary_historical['Rank'].values[0]} (2021) - {direction} by {abs(hungary_change)} positions")
        
        # Armenia comparison
        if not armenia_current.empty and not armenia_historical.empty:
            armenia_change = armenia_historical['Rank'].values[0] - armenia_current['Rank'].values[0]
            direction = "improved" if armenia_change > 0 else "worsened" if armenia_change < 0 else "unchanged"
            print(f"Armenia: {armenia_current['Rank'].values[0]} (current) vs {armenia_historical['Rank'].values[0]} (2021) - {direction} by {abs(armenia_change)} positions")

if __name__ == "__main__":
    main()

=== SCRAPING CURRENT FIFA RANKINGS ===
Accessing: https://football-ranking.com/fifa-rankings
Scraping page 1/4...
Found 50 countries on page 1
Scraping page 2/4...
Found 50 countries on page 2
Scraping page 3/4...
Found 50 countries on page 3
Scraping page 4/4...
Found 50 countries on page 4
Total rankings scraped for current: 200 countries
Data saved to HUN-ARM/data\fifa_rankings_current.xlsx

=== CURRENT RANKINGS ===
Hungary: 41
Armenia: 103
Hungary details: {'Rank': 41, 'Country': 'Hungary', 'Points': 1492.18, 'Previous_Points': '1,501', 'Previous_Rank': '38'}
Armenia details: {'Rank': 103, 'Country': 'Armenia', 'Points': 1219.56, 'Previous_Points': '1,205', 'Previous_Rank': '105'}

=== SCRAPING HISTORICAL FIFA RANKINGS (AUGUST 2021) ===
Accessing: https://football-ranking.com/fifa-rankings?period=12+August+2021
Scraping page 1/4...
Found 50 countries on page 1
Scraping page 2/4...
Found 50 countries on page 2
Scraping page 3/4...
Found 50 countries on page 3
Scraping page 4/4...
Fo

In [20]:
# UEFA Club coefficients

import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_uefa_club_coefficients_kassiesa():
    """UEFA Club Coefficients scraper for kassiesa.net"""
    url = "https://kassiesa.net/uefa/data/method5/trank2025.html"
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        coefficients_data = []
        
        # Find all club rows
        club_rows = soup.find_all('tr', class_='clubline')
        
        for row in club_rows:
            cells = row.find_all(['td', 'th'])
            
            if len(cells) >= 11:  # Ensure we have enough cells
                try:
                    #rank = cells[0].get_text(strip=True)
                    club = cells[2].get_text(strip=True)  # Club name is in the 3rd cell
                    country = cells[3].get_text(strip=True)  # Country code in 4th cell
                    
                    # Points data (different columns represent different seasons)
                    season_2024_25 = cells[4].get_text(strip=True) if len(cells) > 4 else "N/A"
                    season_2023_24 = cells[6].get_text(strip=True) if len(cells) > 6 else "N/A"
                    season_2022_23 = cells[7].get_text(strip=True) if len(cells) > 7 else "N/A"
                    season_2021_22 = cells[8].get_text(strip=True) if len(cells) > 8 else "N/A"
                    
                    # Total points is in the 10th cell (th element with class 'lgray')
                    total_points = cells[9].get_text(strip=True) if len(cells) > 9 else "N/A"
                    
                    # Coefficient for seeding
                    coefficient = cells[10].get_text(strip=True) if len(cells) > 10 else "N/A"
                    
                    if coefficient:
                        coefficients_data.append({
                            #'Rank': int(rank),
                            'Club': club,
                            'Country': country,
                            'Season_2024_25': season_2024_25,
                            'Season_2023_24': season_2023_24,
                            'Season_2022_23': season_2022_23,
                            'Season_2021_22': season_2021_22,
                            'Total_Points': total_points,
                            'Coefficient': coefficient
                        })
                except (ValueError, IndexError, AttributeError) as e:
                    print(f"Hiba a sor feldolgozásakor: {e}")
                    continue
        
        if not coefficients_data:
            print("Nem sikerült UEFA coefficient adatokat találni.")
            return pd.DataFrame()
            
        df = pd.DataFrame(coefficients_data)
        print(f"UEFA Club Coefficients sikeresen lekérve: {len(df)} klub")
        return df
        
    except requests.RequestException as e:
        print(f"Hiba az UEFA oldal lekérésekor: {e}")
        return pd.DataFrame()

def save_xlsx(df, folder="data", filename="uefa_coefficients.xlsx"):
    """Helper function to save DataFrame to Excel"""
    import os
    os.makedirs(folder, exist_ok=True)
    filepath = os.path.join(folder, filename)
    df.to_excel(filepath, index=False)
    print(f"Adatok elmentve: {filepath}")

# UEFA adatok lekérése
uefa_club_df = scrape_uefa_club_coefficients_kassiesa()

if not uefa_club_df.empty:
    # Mentés Excel fájlba
    save_xlsx(uefa_club_df, folder="HUN-ARM/data", filename="uefa_club_coefficients_kassiesa.xlsx")
    
    # Keressük meg a magyar és örmény klubokat
    hungarian_clubs = uefa_club_df[uefa_club_df['Country'].str.contains('HUN', case=False, na=False)]
    armenian_clubs = uefa_club_df[uefa_club_df['Country'].str.contains('ARM', case=False, na=False)]
    
    print(f"\nMagyar klubok az UEFA coefficient listán: {len(hungarian_clubs)}")
    if not hungarian_clubs.empty:
        print(hungarian_clubs[['Club', 'Country', 'Total_Points']].to_string(index=False))
    else:
        print("Nem található magyar klub az UEFA coefficient listán.")
    
    print(f"\nÖrmény klubok az UEFA coefficient listán: {len(armenian_clubs)}")
    if not armenian_clubs.empty:
        print(armenian_clubs[['Club', 'Country', 'Total_Points']].to_string(index=False))
    else:
        print("Nem található örmény klub az UEFA coefficient listán.")
    
    # Összesített információk
    print(f"\nÖsszesítés:")
    print(f"Összes klub: {len(uefa_club_df)}")
    print(f"Legjobb magyar klub: {hungarian_clubs['Club'].iloc[0] if not hungarian_clubs.empty else 'N/A'}")
    print(f"Legjobb örmény klub: {armenian_clubs['Club'].iloc[0] if not armenian_clubs.empty else 'N/A'}")
else:
    print("Nem sikerült adatokat lekérni az UEFA coefficient listáról.")

UEFA Club Coefficients sikeresen lekérve: 427 klub
Adatok elmentve: HUN-ARM/data\uefa_club_coefficients_kassiesa.xlsx

Magyar klubok az UEFA coefficient listán: 10
            Club Country Total_Points
     Ferencváros     Hun       39.000
     FC Fehérvár     Hun        8.000
 Puskás Akadémia     Hun        6.500
        Paksi FC     Hun        2.500
   DVSC Debrecen     Hun        2.000
   Kecskeméti TE     Hun        1.500
Zalaegerszegi TE     Hun        1.500
     Kisvárda FC     Hun        2.000
       Újpest TE     Hun        2.000
 Honvéd Budapest     Hun        1.500

Örmény klubok az UEFA coefficient listán: 7
          Club Country Total_Points
Pyunik Yerevan     Arm        8.500
Ararat-Armenia     Arm        7.500
  Alashkert FC     Arm        6.000
       FC Noah     Arm        5.000
     Urartu FC     Arm        4.000
Ararat Yerevan     Arm        2.500
 Shirak Gyumri     Arm        1.000

Összesítés:
Összes klub: 427
Legjobb magyar klub: Ferencváros
Legjobb örmény klub: P

In [12]:
# FBref

from fbref.fbref_module import scrape

fbref_tables = {
    "ARM_2024-2025_matchlogs": {
        "url": "https://fbref.com/en/squads/95d6caec/2024-2025/matchlogs/c677/schedule/Armenia-Men-Scores-and-Fixtures-UEFA-Nations-League",
        "id": "matchlogs_for"
        },
    "ARM_2024-2025_matchlogs_shooting": {
        "url": "https://fbref.com/en/squads/95d6caec/2024-2025/matchlogs/all_comps/shooting/Armenia-Men-Match-Logs-All-Competitions",
        "id": "matchlogs_for"
        },
    "ARM_2024-2025_matchlogs_shooting_AG": {
        "url": "https://fbref.com/en/squads/95d6caec/2024-2025/matchlogs/all_comps/shooting/Armenia-Men-Match-Logs-All-Competitions",
        "id": "matchlogs_against"
        },
    "ARM_2025_matchlogs": {
        "url": "https://fbref.com/en/squads/95d6caec/2025/matchlogs/c218/schedule/Armenia-Men-Scores-and-Fixtures-Friendlies-M",
        "id": "matchlogs_for"
        },
    "ARM_2025_matchlogs_shooting": {
        "url": "https://fbref.com/en/squads/95d6caec/2025/matchlogs/c218/shooting/Armenia-Men-Match-Logs-Friendlies-M",
        "id": "matchlogs_for"
        },
    "ARM_2025_matchlogs_shooting_AG": {
        "url": "https://fbref.com/en/squads/95d6caec/2025/matchlogs/c218/shooting/Armenia-Men-Match-Logs-Friendlies-M",
        "id": "matchlogs_against"
        },
    "ARM_2026_matchlogs": {
        "url": "https://fbref.com/en/squads/95d6caec/2026/matchlogs/c6/schedule/Armenia-Men-Scores-and-Fixtures-WCQ----UEFA-M",
        "id": "matchlogs_for"
        },
    "ARM_2026_matchlogs_shooting": {
        "url": "https://fbref.com/en/squads/95d6caec/2026/matchlogs/c6/shooting/Armenia-Men-Match-Logs-WCQ----UEFA-M",
        "id": "matchlogs_for"
        },
    "ARM_2026_matchlogs_shooting_AG": {
        "url": "https://fbref.com/en/squads/95d6caec/2026/matchlogs/c6/shooting/Armenia-Men-Match-Logs-WCQ----UEFA-M",
        "id": "matchlogs_against"
        },
}

for name in fbref_tables.keys():
    df = scrape(fbref_tables[name]["url"], fbref_tables[name]["id"])
    save_xlsx(df, folder="HUN-ARM/data", filename=f"fbref_{name}.xlsx")

Mentve ide: HUN-ARM/data\fbref_ARM_2024-2025_matchlogs.xlsx
Mentve ide: HUN-ARM/data\fbref_ARM_2024-2025_matchlogs_shooting.xlsx
Mentve ide: HUN-ARM/data\fbref_ARM_2024-2025_matchlogs_shooting_AG.xlsx
Mentve ide: HUN-ARM/data\fbref_ARM_2025_matchlogs.xlsx
Mentve ide: HUN-ARM/data\fbref_ARM_2025_matchlogs_shooting.xlsx
Mentve ide: HUN-ARM/data\fbref_ARM_2025_matchlogs_shooting_AG.xlsx
Mentve ide: HUN-ARM/data\fbref_ARM_2026_matchlogs.xlsx
Mentve ide: HUN-ARM/data\fbref_ARM_2026_matchlogs_shooting.xlsx
Mentve ide: HUN-ARM/data\fbref_ARM_2026_matchlogs_shooting_AG.xlsx


In [3]:
# SofaScore functions

import pandas as pd
import json

def scrape_sofascore(url):
    import tls_client
    import time
    import random

    # Válassz "client_profile"-t ami Chrome/Safari-szerű fingerprintet ad.
    sess = tls_client.Session(client_identifier="chrome_118")

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                    "(KHTML, like Gecko) Chrome/118.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Referer": "https://www.sofascore.com/",
        "Origin": "https://www.sofascore.com",
    }

    #time.sleep(random.randint(1,3))

    resp = sess.get(url, headers=headers)

    if resp.status_code == 200:
        data = resp.json()

        return data

    else:
        print(f"Error: {resp.status_code}")
        return {}


# 1. Lineups DataFrame
def create_lineups_df(lineups_data):
    home_players = []
    away_players = []
    
    # Process home players
    for player in lineups_data['home']['players']:
        player_info = player['player'].copy()
        if 'statistics' in player:
            player_info.update(player['statistics'])
        player_info['team'] = 'home'
        player_info['substitute'] = player['substitute']
        player_info['captain'] = player.get('captain', False)
        home_players.append(player_info)
    
    # Process away players
    for player in lineups_data['away']['players']:
        player_info = player['player'].copy()
        if 'statistics' in player:
            player_info.update(player['statistics'])
        player_info['team'] = 'away'
        player_info['substitute'] = player['substitute']
        player_info['captain'] = player.get('captain', False)
        away_players.append(player_info)
    
    # Combine both teams
    all_players = home_players + away_players
    lineups_df = pd.DataFrame(all_players)
    
    return lineups_df

# 2. Average Positions DataFrame
def create_average_positions_df(average_positions_data):
    home_positions = []
    away_positions = []
    
    # Process home players
    for player in average_positions_data['home']:
        player_info = player['player'].copy()
        player_info.update({
            'averageX': player['averageX'],
            'averageY': player['averageY'],
            'pointsCount': player['pointsCount'],
            'team': 'home'
        })
        home_positions.append(player_info)
    
    # Process away players
    for player in average_positions_data['away']:
        player_info = player['player'].copy()
        player_info.update({
            'averageX': player['averageX'],
            'averageY': player['averageY'],
            'pointsCount': player['pointsCount'],
            'team': 'away'
        })
        away_positions.append(player_info)
    
    # Combine both teams
    all_positions = home_positions + away_positions
    positions_df = pd.DataFrame(all_positions)
    
    return positions_df

# 3. Statistics DataFrame
def create_statistics_df(statistics_data):
    all_stats = []
    
    for period_data in statistics_data['statistics']:
        period = period_data['period']
        
        for group in period_data['groups']:
            group_name = group['groupName']
            
            for item in group['statisticsItems']:
                stat_info = {
                    'period': period,
                    'group': group_name,
                    'statistic': item['name'],
                    'home_value': item.get('homeValue'),
                    'away_value': item.get('awayValue'),
                    'home_total': item.get('homeTotal'),
                    'away_total': item.get('awayTotal'),
                    'home_display': item.get('home'),
                    'away_display': item.get('away'),
                    'compare_code': item.get('compareCode'),
                    'key': item.get('key')
                }
                all_stats.append(stat_info)
    
    return pd.DataFrame(all_stats)

# 4. Shotmap DataFrame
def create_shotmap_df(shotmap_data):
    shots = []
    
    for shot in shotmap_data['shotmap']:
        shot_info = shot['player'].copy()
        shot_info.update({
            'isHome': shot['isHome'],
            'shotType': shot['shotType'],
            'situation': shot['situation'],
            'bodyPart': shot['bodyPart'],
            'time': shot['time'],
            'timeSeconds': shot['timeSeconds'],
            'periodTimeSeconds': shot.get('periodTimeSeconds'),
            'goalType': shot.get('goalType'),
            'playerX': shot['playerCoordinates']['x'],
            'playerY': shot['playerCoordinates']['y'],
            'goalMouthX': shot['goalMouthCoordinates']['x'] if 'goalMouthCoordinates' in shot else None,
            'goalMouthY': shot['goalMouthCoordinates']['y'] if 'goalMouthCoordinates' in shot else None,
            'goalMouthZ': shot['goalMouthCoordinates']['z'] if 'goalMouthCoordinates' in shot else None
        })
        shots.append(shot_info)
    
    return pd.DataFrame(shots)

# 5. Graph DataFrame
def create_graph_df(graph_data):
    graph_points = []
    
    for point in graph_data['graphPoints']:
        graph_points.append({
            'minute': point['minute'],
            'value': point['value']
        })
    
    return pd.DataFrame(graph_points)

# 6. Player stats DataFrame
def create_player_stats_df(player_stats_data):
    """
    Convert player statistics data to a pandas DataFrame
    
    Parameters:
    player_stats_data (list): List of player statistics dictionaries
    
    Returns:
    pd.DataFrame: DataFrame containing player statistics
    """
    all_player_stats = []
    
    for player_data in player_stats_data:
        # Alap játékos információk
        player_info = player_data['player'].copy()
        
        # Csapat információk hozzáadása
        player_info['team_name'] = player_data['team']['name']
        player_info['team_id'] = player_data['team']['id']
        
        # Pozíció hozzáadása
        player_info['position'] = player_data.get('position', '')
        
        # Statisztikák hozzáadása, ha vannak
        if player_data['statistics']:
            player_info.update(player_data['statistics'])
        
        all_player_stats.append(player_info)
    
    # DataFrame létrehozása
    player_stats_df = pd.DataFrame(all_player_stats)
    
    return player_stats_df

# 7. Game odds
def create_odds_df(odds_data):
    odds_list = []
    if odds_data["markets"]:
        for market in odds_data['markets']:
            if (market['marketGroup'] == '1X2') and (market['marketName'] == 'Full time'):
                for choice in market['choices']:
                    dividend, divisor = choice["fractionalValue"].split('/') 
                    odds = int(dividend) / int(divisor) + 1
                    odds_list.append({
                        "name": choice["name"],
                        "odds": odds,
                        "prob": 1/odds
                    })

        df_odds = pd.DataFrame(odds_list)
        df_odds['prob_corr'] = df_odds['prob'] / df_odds['prob'].sum()
        
        return df_odds
    else:
        return pd.DataFrame()

In [5]:
# SofaScore JSONs

call_list = ['lineups', 'average-positions', 'statistics', 'shotmap', 'graph']

event_ids = {
    "ARM-IRL": 13233581,
    "ARM-POR": 13233472,
    "GEO-ARM": 13157439,
    "ARM-GEO": 13157435,
    "LVA-ARM": 12057793,
    "ARM-FRO": 12057795,
    "ARM-MKD": 12057796,
    "FRO-ARM": 12057797,
    "MKD-ARM": 12057800,
    "ARM-LVA": 12057799
}

for event_name, event_id in event_ids.items():
    for call in call_list:
        print(f"\n\n{event_name} - {call}")
        url = f"https://www.sofascore.com/api/v1/event/{event_id}/{call}"
        data = scrape_sofascore(url)

        # Create appropriate DataFrame based on the endpoint
        if call == 'lineups':
            lineups_df = create_lineups_df(data)
            print(f"Lineups DataFrame shape: {lineups_df.shape}")
            
        elif call == 'average-positions':
            positions_df = create_average_positions_df(data)
            print(f"Positions DataFrame shape: {positions_df.shape}")
            
        elif call == 'statistics':
            stats_df = create_statistics_df(data)
            print(f"Statistics DataFrame shape: {stats_df.shape}")
            
        elif call == 'shotmap':
            shotmap_df = create_shotmap_df(data)
            print(f"Shotmap DataFrame shape: {shotmap_df.shape}")
            
        elif call == 'graph':
            graph_df = create_graph_df(data)
            print(f"Graph DataFrame shape: {graph_df.shape}")

        elif call == 'odds':
            graph_df = create_odds_df(data)
            print(f"Graph DataFrame shape: {graph_df.shape}")

    # Get stats for each player
    player_stats_list = [] 

    for player_id in positions_df.id.unique():
        url_player = f"https://www.sofascore.com/api/v1/event/{event_id}/player/{player_id}/statistics"
        data_player = scrape_sofascore(url_player)
        
        if data_player:  # Ha sikeres volt a lekérés
            player_stats_list.append(data_player)

    # DataFrame létrehozása
    player_stats_df = create_player_stats_df(player_stats_list)
    # Eredmény megjelenítése
    print(f"Player stats DataFrame shape: {player_stats_df.shape}")

    # Odds data
    odds_data = scrape_sofascore(f"https://www.sofascore.com/api/v1/event/{event_id}/odds/1/all")
    odds_df = create_odds_df(odds_data)
    print(f"Odds DataFrame shape: {odds_df.shape}")

    all_data = {
        'Lineups': lineups_df,
        'Positions': positions_df,
        'Statistics': stats_df,
        'Shotmap': shotmap_df,
        'Timeline': graph_df, 
        'Player_Stats': player_stats_df,
        'Odds': odds_df
    }
    save_xlsx(all_data, "HUN-ARM/data", f"sofascore_{event_name}.xlsx")



ARM-IRL - lineups
Lineups DataFrame shape: (46, 62)


ARM-IRL - average-positions
Positions DataFrame shape: (30, 15)


ARM-IRL - statistics
Statistics DataFrame shape: (121, 11)


ARM-IRL - shotmap
Shotmap DataFrame shape: (23, 24)


ARM-IRL - graph
Graph DataFrame shape: (92, 2)
Player stats DataFrame shape: (30, 60)
Odds DataFrame shape: (3, 4)
Munkalap mentve: 'Lineups'
Munkalap mentve: 'Positions'
Munkalap mentve: 'Statistics'
Munkalap mentve: 'Shotmap'
Munkalap mentve: 'Timeline'
Munkalap mentve: 'Player_Stats'
Munkalap mentve: 'Odds'
Mentve ide: HUN-ARM/data\sofascore_ARM-IRL.xlsx


ARM-POR - lineups
Lineups DataFrame shape: (46, 59)


ARM-POR - average-positions
Positions DataFrame shape: (32, 15)


ARM-POR - statistics
Statistics DataFrame shape: (118, 11)


ARM-POR - shotmap
Shotmap DataFrame shape: (31, 24)


ARM-POR - graph
Graph DataFrame shape: (91, 2)
Player stats DataFrame shape: (32, 57)
Odds DataFrame shape: (3, 4)
Munkalap mentve: 'Lineups'
Munkalap mentve: 'Positi

In [39]:
# Transfermarkt


# CLEAN DATA

In [24]:
# Load Fbref data

import pandas as pd
import os
import glob

def load_pattern_data(data_folder='HUN-ARM/data', pattern="fbref_ARM_*shooting.xlsx"):
    """
    Betölti az összes 'fbref_ARM_' kezdetű és 'shooting' tartalmú fájlt
    egyetlen nagy DataFrame-be
    """
    # Összes megfelelő fájl keresése
    file_pattern = os.path.join(data_folder, pattern)
    matching_files = glob.glob(file_pattern)
    
    if not matching_files:
        print("Nincsenek megfelelő fájlok a mappában")
        return pd.DataFrame()
    
    print(f"Talált fájlok: {len(matching_files)}")
    
    # Összes fájl betöltése és összefűzése
    all_dataframes = []
    
    for file_path in matching_files:
        try:
            result = load_xlsx(file_path)
            
            # Ha dictionary-t kapunk, nézzük meg melyik tartalmaz DataFrame-eket
            if isinstance(result, dict):
                # Válasszuk ki az első DataFrame-et a dictionary-ből
                # vagy keressük meg a 'shooting' táblát
                for key, value in result.items():
                    if isinstance(value, pd.DataFrame):
                        df = value
                        df['source_file'] = os.path.basename(file_path)
                        df['sheet_name'] = key  # Opcionális: tábla neve
                        all_dataframes.append(df)
                        print(f"Betöltve: {os.path.basename(file_path)} - {key}")
                        break
            elif isinstance(result, pd.DataFrame):
                # Ha közvetlenül DataFrame-et kapunk
                result['source_file'] = os.path.basename(file_path)
                all_dataframes.append(result)
                print(f"Betöltve: {os.path.basename(file_path)}")
            else:
                print(f"Egyéb típus: {type(result)} - {os.path.basename(file_path)}")
                
        except Exception as e:
            print(f"Hiba a {file_path} betöltésekor: {e}")
    
    # Összes DataFrame összefűzése
    if all_dataframes:
        final_df = pd.concat(all_dataframes, ignore_index=True)
        print(f"Összesen {len(final_df)} sor betöltve")
        return final_df
    else:
        print("Nem sikerült DataFrame-eket betölteni")
        return pd.DataFrame()

# Használat
matchlogs_df = load_pattern_data(pattern="fbref_ARM_*matchlogs.xlsx")
shooting_for_df = load_pattern_data(pattern="fbref_ARM_*shooting.xlsx")
shooting_ag_df = load_pattern_data(pattern="fbref_ARM_*shooting_AG.xlsx")


Talált fájlok: 3
Betöltve: HUN-ARM/data\fbref_ARM_2024-2025_matchlogs.xlsx (1 munkalap)
Betöltve: fbref_ARM_2024-2025_matchlogs.xlsx - Sheet1
Betöltve: HUN-ARM/data\fbref_ARM_2025_matchlogs.xlsx (1 munkalap)
Betöltve: fbref_ARM_2025_matchlogs.xlsx - Sheet1
Betöltve: HUN-ARM/data\fbref_ARM_2026_matchlogs.xlsx (1 munkalap)
Betöltve: fbref_ARM_2026_matchlogs.xlsx - Sheet1
Összesen 17 sor betöltve
Talált fájlok: 3
Betöltve: HUN-ARM/data\fbref_ARM_2024-2025_matchlogs_shooting.xlsx (1 munkalap)
Betöltve: fbref_ARM_2024-2025_matchlogs_shooting.xlsx - Sheet1
Betöltve: HUN-ARM/data\fbref_ARM_2025_matchlogs_shooting.xlsx (1 munkalap)
Betöltve: fbref_ARM_2025_matchlogs_shooting.xlsx - Sheet1
Betöltve: HUN-ARM/data\fbref_ARM_2026_matchlogs_shooting.xlsx (1 munkalap)
Betöltve: fbref_ARM_2026_matchlogs_shooting.xlsx - Sheet1
Összesen 15 sor betöltve
Talált fájlok: 3
Betöltve: HUN-ARM/data\fbref_ARM_2024-2025_matchlogs_shooting_AG.xlsx (1 munkalap)
Betöltve: fbref_ARM_2024-2025_matchlogs_shooting_AG.

In [25]:
sofascore_a = load_pattern_data(pattern="sofascore_*.xlsx")

Talált fájlok: 10
Betöltve: HUN-ARM/data\sofascore_ARM-FRO.xlsx (7 munkalap)
Betöltve: sofascore_ARM-FRO.xlsx - Lineups
Betöltve: HUN-ARM/data\sofascore_ARM-GEO.xlsx (7 munkalap)
Betöltve: sofascore_ARM-GEO.xlsx - Lineups
Betöltve: HUN-ARM/data\sofascore_ARM-IRL.xlsx (7 munkalap)
Betöltve: sofascore_ARM-IRL.xlsx - Lineups
Betöltve: HUN-ARM/data\sofascore_ARM-LVA.xlsx (7 munkalap)
Betöltve: sofascore_ARM-LVA.xlsx - Lineups
Betöltve: HUN-ARM/data\sofascore_ARM-MKD.xlsx (7 munkalap)
Betöltve: sofascore_ARM-MKD.xlsx - Lineups
Betöltve: HUN-ARM/data\sofascore_ARM-POR.xlsx (7 munkalap)
Betöltve: sofascore_ARM-POR.xlsx - Lineups
Betöltve: HUN-ARM/data\sofascore_FRO-ARM.xlsx (7 munkalap)
Betöltve: sofascore_FRO-ARM.xlsx - Lineups
Betöltve: HUN-ARM/data\sofascore_GEO-ARM.xlsx (7 munkalap)
Betöltve: sofascore_GEO-ARM.xlsx - Lineups
Betöltve: HUN-ARM/data\sofascore_LVA-ARM.xlsx (7 munkalap)
Betöltve: sofascore_LVA-ARM.xlsx - Lineups
Betöltve: HUN-ARM/data\sofascore_MKD-ARM.xlsx (7 munkalap)
Betöl

In [27]:
t = load_xlsx("HUN-ARM/data/sofascore_MKD-ARM.xlsx")

Betöltve: HUN-ARM/data/sofascore_MKD-ARM.xlsx (7 munkalap)


In [28]:
t['Lineups']

,name,firstName,lastName,slug,shortName,position,jerseyNumber,height,userCount,id,...,totalOffside,bigChanceCreated,bigChanceMissed,goals,blockedScoringAttempt,errorLeadToAGoal,totalKeeperSweeper,accurateKeeperSweeper,lastManTackle,hitWoodwork
0,Stole Dimitrievski,NaN,NaN,stole-dimitrievski,S. Dimitrievski,G,1.0,188,808,97951,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Stefan Aškovski,NaN,NaN,stefan-askovski,S. Aškovski,M,NaN,179,68,99533,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Gjoko Zajkov,NaN,NaN,gjoko-zajkov,G. Zajkov,D,5.0,186,117,190239,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Visar Musliu,NaN,NaN,visar-musliu,V. Musliu,D,26.0,186,180,134231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Bojan Dimoski,NaN,NaN,bojan-dimoski,B. Dimoski,D,20.0,176,151,1005774,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,David Babunski,NaN,NaN,david-babunski,D. Babunski,M,NaN,176,118,134235,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Isnik Alimi,NaN,NaN,isnik-alimi,I. Alimi,M,4.0,186,136,166309,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Darko Churlinov,NaN,NaN,darko-churlinov,D. Churlinov,M,17.0,180,526,876345,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Enis Bardhi,NaN,NaN,enis-bardhi,E. Bardhi,M,10.0,172,1080,355024,...,NaN,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
9,Eljif Elmas,NaN,NaN,eljif-elmas,E. Elmas,M,20.0,182,3374,838232,...,NaN,NaN,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN
